# Lab 10: Fuzzy logika

Koristi se `scikit-fuzzy` paket za kreiranje fuzzy sistema upravljanja.


In [ ]:
# Instalacija potrebnih paketa (automatski preskace ako su vec instalirani)
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("scikit-fuzzy", "skfuzzy")
ensure("scipy")
ensure("networkx")
print("Svi paketi su instalirani.")


In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt
print("scikit-fuzzy verzija:", fuzz.__version__)


## Zadatak 1 - Fuzzy sistem za iznos napojnice

**Ulazi:** kvalitet hrane i usluge (0-10)  
**Izlaz:** napojnica (0-25%)  
**Pravila:**
- IF usluga IS good OR hrana IS good THEN napojnica IS velika
- IF usluga IS average THEN napojnica IS srednja
- IF usluga IS poor AND hrana IS poor THEN napojnica IS mala


### a) Definisanje ulaza i izlaza


In [ ]:
# Definisanje ulaza i izlaza
hrana = ctrl.Antecedent(np.arange(0, 11, 1), 'hrana')
usluga = ctrl.Antecedent(np.arange(0, 11, 1), 'usluga')
napojnica = ctrl.Consequent(np.arange(0, 26, 1), 'napojnica')

print("Ulazi i izlaz definirani:")
print(f"  hrana:    {hrana.universe[0]}-{hrana.universe[-1]}")
print(f"  usluga:   {usluga.universe[0]}-{usluga.universe[-1]}")
print(f"  napojnica: {napojnica.universe[0]}-{napojnica.universe[-1]}%")


### b) Definisanje funkcija pripadnosti


In [ ]:
# Automatsko dodavanje funkcija pripadnosti za ulaze (poor, average, good)
hrana.automf(3)
usluga.automf(3)

# Rucno pravljenje trougaonih funkcija pripadnosti za izlaz
napojnica['mala']   = fuzz.trimf(napojnica.universe, [0, 0, 13])
napojnica['srednja'] = fuzz.trimf(napojnica.universe, [0, 13, 25])
napojnica['velika']  = fuzz.trimf(napojnica.universe, [13, 25, 25])

# Vizualizacija funkcija pripadnosti
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].set_title('Hrana (ulaz)')
for term in ['poor', 'average', 'good']:
    axes[0].plot(hrana.universe, hrana[term].mf, label=term)
axes[0].set_xlabel('Ocjena (0-10)'); axes[0].set_ylabel('Stepen pripadnosti')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Usluga (ulaz)')
for term in ['poor', 'average', 'good']:
    axes[1].plot(usluga.universe, usluga[term].mf, label=term)
axes[1].set_xlabel('Ocjena (0-10)')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].set_title('Napojnica (izlaz)')
for term in ['mala', 'srednja', 'velika']:
    axes[2].plot(napojnica.universe, napojnica[term].mf, label=term)
axes[2].set_xlabel('Napojnica (%)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lab10_mf1.png', dpi=80)
plt.show()
print("Funkcije pripadnosti vizualizirane.")


### c) Definisanje pravila


In [ ]:
# Definisanje pravila
p1 = ctrl.Rule(hrana['poor'] | usluga['poor'], napojnica['mala'])
p2 = ctrl.Rule(usluga['average'], napojnica['srednja'])
p3 = ctrl.Rule(usluga['good'] | hrana['good'], napojnica['velika'])

print("Pravila definirana:")
print("  p1: IF hrana IS poor OR usluga IS poor THEN napojnica IS mala")
print("  p2: IF usluga IS average THEN napojnica IS srednja")
print("  p3: IF usluga IS good OR hrana IS good THEN napojnica IS velika")


### d) Kreiranje sistema upravljanja


In [ ]:
# Kreiranje sistema upravljanja
sistem = ctrl.ControlSystem([p1, p2, p3])
print("Sistem upravljanja kreiran.")


### e) Prevod u simulaciju


In [ ]:
# Prevod u simulaciju
sistemSim = ctrl.ControlSystemSimulation(sistem)
print("Simulacija pripremljena.")


### f) Pokretanje simulacije


In [ ]:
# Zadavanje crisp ulaza
sistemSim.input['hrana'] = 6.5
sistemSim.input['usluga'] = 9.8

# Racunanje izlaza
sistemSim.compute()
rezultat = sistemSim.output['napojnica']
print(f"Hrana: 6.5/10, Usluga: 9.8/10")
print(f"Preporucena napojnica: {rezultat:.2f}%")


### Vizualizacija rezultata - Zadatak 1


In [ ]:
napojnica.view(sim=sistemSim)
plt.title('Fuzzy izlaz - napojnica (Zadatak 1)')
plt.savefig('lab10_result1.png', dpi=80)
plt.show()


### Testiranje vise vrijednosti


In [ ]:
test_cases = [
    (2.0, 1.0, "Losa hrana, losa usluga"),
    (5.0, 5.0, "Prosjecna hrana, prosjecna usluga"),
    (9.0, 9.5, "Odlicna hrana, odlicna usluga"),
    (6.5, 9.8, "Dobra hrana, odlicna usluga (originalni primjer)"),
]

print(f"{'Hrana':>6} {'Usluga':>7} {'Napojnica':>10}  Opis")
print("-" * 60)
for h, u, opis in test_cases:
    sim = ctrl.ControlSystemSimulation(sistem)
    sim.input['hrana'] = h
    sim.input['usluga'] = u
    sim.compute()
    tip = sim.output['napojnica']
    print(f"{h:>6.1f} {u:>7.1f} {tip:>9.2f}%  {opis}")


## Zadatak 2 - Fuzzy sistem sa max napojnicom 50%

Iste kao Zadatak 1, ali:
- Maksimalna napojnica je **50%** (ne 25%)
- Funkcije pripadnosti za ulaze su **rucno definirane** (ne automf)
- Za 'prihvatljiva' usluga i hrana koristi se **trapezoidna** funkcija sa korijenom na [4, 6]
- Ostale funkcije su trougaone


In [ ]:
# Definisanje ulaza i izlaza - max napojnica 50%
hrana2   = ctrl.Antecedent(np.arange(0, 11, 1), 'hrana')
usluga2  = ctrl.Antecedent(np.arange(0, 11, 1), 'usluga')
napojnica2 = ctrl.Consequent(np.arange(0, 51, 1), 'napojnica')

# --- Rucno definirane funkcije pripadnosti za ulaze ---
# Losa (poor):  trougaona, [0, 0, 5]
# Prihvatljiva (average): trapezoidna, korijen na [4, 6] => [2, 4, 6, 8]
# Odlicna (good): trougaona, [5, 10, 10]

hrana2['poor']    = fuzz.trimf(hrana2.universe,   [0, 0,  5])
hrana2['average'] = fuzz.trapmf(hrana2.universe,  [2, 4,  6, 8])
hrana2['good']    = fuzz.trimf(hrana2.universe,   [5, 10, 10])

usluga2['poor']    = fuzz.trimf(usluga2.universe,  [0, 0,  5])
usluga2['average'] = fuzz.trapmf(usluga2.universe, [2, 4,  6, 8])
usluga2['good']    = fuzz.trimf(usluga2.universe,  [5, 10, 10])

# Trougaone funkcije za izlaz (skaliran na 0-50%)
napojnica2['mala']   = fuzz.trimf(napojnica2.universe, [0,  0,  25])
napojnica2['srednja'] = fuzz.trimf(napojnica2.universe, [0,  25, 50])
napojnica2['velika']  = fuzz.trimf(napojnica2.universe, [25, 50, 50])

print("Ulazi i izlaz za Zadatak 2 definirani.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].set_title('Hrana - rucne MF (Zad. 2)')
for term, ls in zip(['poor', 'average', 'good'], ['-', '--', '-.']):
    axes[0].plot(hrana2.universe, hrana2[term].mf, ls, label=term, linewidth=2)
axes[0].set_xlabel('Ocjena (0-10)'); axes[0].set_ylabel('Stepen pripadnosti')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Usluga - rucne MF (Zad. 2)')
for term, ls in zip(['poor', 'average', 'good'], ['-', '--', '-.']):
    axes[1].plot(usluga2.universe, usluga2[term].mf, ls, label=term, linewidth=2)
axes[1].set_xlabel('Ocjena (0-10)')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].set_title('Napojnica (0-50%) - Zad. 2')
for term, ls in zip(['mala', 'srednja', 'velika'], ['-', '--', '-.']):
    axes[2].plot(napojnica2.universe, napojnica2[term].mf, ls, label=term, linewidth=2)
axes[2].set_xlabel('Napojnica (%)')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('lab10_mf2.png', dpi=80)
plt.show()

# Prikaz razlike: trapezoidna vs trougaona za 'average'
fig2, ax = plt.subplots(figsize=(8, 4))
ax.plot(usluga.universe, usluga['average'].mf, 'b-', label='automf (trougaona)', linewidth=2)
ax.plot(usluga2.universe, usluga2['average'].mf, 'r--', label='trapmf korijen [4,6]', linewidth=2)
ax.set_title('Usporedba: trougaona vs trapezoidna za "average"')
ax.set_xlabel('Ocjena usluge (0-10)'); ax.set_ylabel('Stepen pripadnosti')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lab10_compare_mf.png', dpi=80)
plt.show()


In [ ]:
# Pravila - ista logika
q1 = ctrl.Rule(hrana2['poor']    | usluga2['poor'],    napojnica2['mala'])
q2 = ctrl.Rule(usluga2['average'],                     napojnica2['srednja'])
q3 = ctrl.Rule(usluga2['good']   | hrana2['good'],     napojnica2['velika'])

sistem2 = ctrl.ControlSystem([q1, q2, q3])
sistemSim2 = ctrl.ControlSystemSimulation(sistem2)

# Test sa istim ulazima kao Zadatak 1
sistemSim2.input['hrana'] = 6.5
sistemSim2.input['usluga'] = 9.8
sistemSim2.compute()

rezultat2 = sistemSim2.output['napojnica']
print(f"Zadatak 2 - Hrana: 6.5/10, Usluga: 9.8/10")
print(f"Preporucena napojnica (max 50%): {rezultat2:.2f}%")


In [ ]:
napojnica2.view(sim=sistemSim2)
plt.title('Fuzzy izlaz - napojnica (Zadatak 2, max 50%)')
plt.savefig('lab10_result2.png', dpi=80)
plt.show()


### Usporedba Zadatka 1 i Zadatka 2


In [ ]:
test_cases = [
    (1.0, 1.0),
    (3.0, 3.0),
    (5.0, 5.0),
    (6.5, 6.5),
    (6.5, 9.8),
    (9.0, 9.0),
    (10.0, 10.0),
]

print(f"{'Hrana':>6} {'Usluga':>7} {'Nap.1 (0-25%)':>14} {'Nap.2 (0-50%)':>14}")
print("-" * 50)
for h, u in test_cases:
    s1 = ctrl.ControlSystemSimulation(sistem)
    s1.input['hrana'] = h; s1.input['usluga'] = u; s1.compute()
    s2 = ctrl.ControlSystemSimulation(sistem2)
    s2.input['hrana'] = h; s2.input['usluga'] = u; s2.compute()
    print(f"{h:>6.1f} {u:>7.1f} {s1.output['napojnica']:>13.2f}% {s2.output['napojnica']:>13.2f}%")
